<a href="https://colab.research.google.com/github/Him-an-shi/Seminar/blob/main/Experiment_01_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# EXPERIMENT 1 — Environment Check
# This cell does not download any dataset.

import sys
import os
import platform
import shutil

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

# Check available disk space in the current environment
total, used, free = shutil.disk_usage("/content")
print(f"Free notebook disk space: {free / (1024**3):.2f} GB")

# Check PyTorch
try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Selected device:", device)
except ImportError:
    print("PyTorch is not installed.")

# Check basic numerical libraries
try:
    import numpy
    print("NumPy version:", numpy.__version__)
except ImportError:
    print("NumPy is not installed.")


Python version: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Free notebook disk space: 87.21 GB
PyTorch version: 2.11.0+cpu
CUDA available: False
Selected device: cpu
NumPy version: 2.1.3


In [ ]:

# EXPERIMENT 1: Single-task training smoke test
# Small built-in image dataset; no large downloads.

import sys
import time
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
except ImportError as e:
    print("Missing package:", e)
    print("Run this in a separate cell: %pip install torch scikit-learn")
    raise

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

# Load 8x8 images (no dataset download)
data = load_digits()
X = data.images.astype(np.float32) / 16.0
y = data.target.astype(np.int64)

# Fixed train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Convert images to tensors
X_train = torch.tensor(X_train).reshape(-1, 64)
X_test = torch.tensor(X_test).reshape(-1, 64)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)

# Small neural network
class SmallClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)

model = SmallClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train on one task
EPOCHS = 8
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader.dataset):.4f}"
    )

# Evaluate on held-out test data
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        predictions = outputs.argmax(dim=1)
        correct += (predictions.cpu() == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
elapsed = time.time() - start

print("\n--- EXPERIMENT 1 RESULTS ---")
print(f"Training images: {len(X_train)}")
print(f"Test images: {len(X_test)}")
print(f"Test accuracy: {accuracy:.2f}%")
print(f"Training time: {elapsed:.2f} seconds")

# Save this test run's model and metrics in the notebook workspace
torch.save(model.state_dict(), "/content/experiment1_smoke_test.pt")

with open("/content/experiment1_smoke_test.txt", "w") as f:
    f.write(f"Dataset: sklearn Digits (pipeline test only)\n")
    f.write(f"Seed: {SEED}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write(f"Test accuracy: {accuracy:.2f}%\n")
    f.write(f"Training time: {elapsed:.2f} seconds\n")

print("Test model and metrics saved in /content/")


PyTorch: 2.11.0+cpu
Device: cpu
Epoch 1/8 | Loss: 2.1779
Epoch 2/8 | Loss: 1.7923
Epoch 3/8 | Loss: 1.3037
Epoch 4/8 | Loss: 0.9112
Epoch 5/8 | Loss: 0.6630
Epoch 6/8 | Loss: 0.5122
Epoch 7/8 | Loss: 0.4197
Epoch 8/8 | Loss: 0.3555

--- EXPERIMENT 1 RESULTS ---
Training images: 1437
Test images: 360
Test accuracy: 93.33%
Training time: 0.59 seconds
Test model and metrics saved in /content/


In [ ]:

# Research baseline: CIFAR-10, Task A
# CPU-friendly subset; no pretrained model required.

import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print("Device:", device)

# Download CIFAR-10 to Colab's temporary storage
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=True,
    transform=transform
)
test_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=False,
    download=True,
    transform=transform
)

# CIFAR-10 labels:
# 0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer,
# 5 dog, 6 frog, 7 horse, 8 ship, 9 truck
TASK_A = [0, 1, 2, 3, 4]
PER_CLASS_TRAIN = 500
PER_CLASS_TEST = 100

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    chosen = []
    for c in classes:
        indices = np.where(targets == c)[0]
        rng.shuffle(indices)
        chosen.extend(indices[:per_class].tolist())
    return chosen

train_indices = choose_indices(
    train_data, TASK_A, PER_CLASS_TRAIN, SEED
)
test_indices = choose_indices(
    test_data, TASK_A, PER_CLASS_TEST, SEED
)

train_loader = DataLoader(
    Subset(train_data, train_indices),
    batch_size=32,
    shuffle=True,
    num_workers=2
)
test_loader = DataLoader(
    Subset(test_data, test_indices),
    batch_size=64,
    shuffle=False,
    num_workers=2
)

# Small CNN for 32x32 images
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        # Remap original labels 0-4 to task labels 0-4
        labels = labels.to(device)
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_indices):.4f}"
    )

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        predictions = outputs.argmax(dim=1)
        correct += (predictions.cpu() == labels).sum().item()
        total += labels.numel()

accuracy = 100 * correct / total
elapsed = time.time() - start

print("\n--- CIFAR-10 TASK A RESULTS ---")
print("Classes:", TASK_A)
print("Training images:", len(train_indices))
print("Test images:", len(test_indices))
print(f"Test accuracy: {accuracy:.2f}%")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp1_cifar10_taskA.pt")

with open("/content/exp1_cifar10_taskA.txt", "w") as f:
    f.write(f"Seed: {SEED}\n")
    f.write(f"Classes: {TASK_A}\n")
    f.write(f"Train images: {len(train_indices)}\n")
    f.write(f"Test images: {len(test_indices)}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write(f"Accuracy: {accuracy:.2f}%\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Checkpoint and metrics saved.")


Device: cpu
Epoch 1/5 | Loss: 1.3567
Epoch 2/5 | Loss: 1.0545
Epoch 3/5 | Loss: 0.9053
Epoch 4/5 | Loss: 0.8303
Epoch 5/5 | Loss: 0.7600

--- CIFAR-10 TASK A RESULTS ---
Classes: [0, 1, 2, 3, 4]
Training images: 2500
Test images: 500
Test accuracy: 63.40%
Training time: 12.2 seconds
Checkpoint and metrics saved.


In [ ]:

import os
import shutil
import time

print("Checking dataset download...")

dataset_path = "/content/cifar10/cifar-10-batches-py"

if os.path.exists(dataset_path):
    print("Dataset folder exists.")
    print("Files:", os.listdir(dataset_path)[:10])
else:
    print("Dataset folder is not complete or not present.")

total, used, free = shutil.disk_usage("/content")
print(f"Free runtime disk: {free / (1024**3):.2f} GB")


Checking dataset download...
Dataset folder exists.
Files: ['data_batch_1', 'data_batch_4', 'readme.html', 'data_batch_3', 'test_batch', 'data_batch_5', 'data_batch_2', 'batches.meta']
Free runtime disk: 86.84 GB


In [ ]:

# Backup Experiment 1B before sequential training
import os
import shutil

checkpoint = "/content/exp1_cifar10_taskA.pt"
metrics = "/content/exp1_cifar10_taskA.txt"

print("Checkpoint exists:", os.path.exists(checkpoint))
print("Metrics file exists:", os.path.exists(metrics))

if os.path.exists(checkpoint):
    shutil.copy2(
        checkpoint,
        "/content/exp1_cifar10_taskA_backup.pt"
    )
    print("Original model checkpoint backed up.")

if os.path.exists(metrics):
    print("\nBaseline metrics:")
    with open(metrics, "r") as f:
        print(f.read())


Checkpoint exists: True
Metrics file exists: True
Original model checkpoint backed up.

Baseline metrics:
Seed: 42
Classes: [0, 1, 2, 3, 4]
Train images: 2500
Test images: 500
Epochs: 5
Accuracy: 63.40%
Training time: 12.2 seconds



In [ ]:

# EXPERIMENT 2: Sequential learning WITHOUT replay
# Task A -> Task B, then evaluate both tasks.

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cpu")

# Same dataset and preprocessing as Experiment 1B
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=False,
    transform=transform
)
test_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=False,
    download=False,
    transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    chosen = []
    for c in classes:
        indices = np.where(targets == c)[0]
        rng.shuffle(indices)
        chosen.extend(indices[:per_class].tolist())
    return chosen

# Reproduce the same subsets from Experiment 1B
train_A = choose_indices(train_data, TASK_A, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def make_loader(dataset, indices, shuffle):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

loader_A = make_loader(train_data, train_A, True)
loader_B = make_loader(train_data, train_B, True)
test_loader_A = make_loader(test_data, test_A, False)
test_loader_B = make_loader(test_data, test_B, False)

# Identical architecture to Experiment 1B
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            # Map global labels to task-local labels 0-4
            labels = labels - 5 if loader is test_loader_B else labels
            outputs = model(images.to(device))
            predictions = outputs.argmax(dim=1).cpu()
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return 100.0 * correct / total

# Load original Task A model and confirm its baseline score
model = SmallCNN().to(device)
model.load_state_dict(
    torch.load(
        "/content/exp1_cifar10_taskA_backup.pt",
        map_location=device,
        weights_only=True
    )
)

# Check Task A before learning B
accuracy_A_before = evaluate(model, test_loader_A)
print(f"Task A accuracy before Task B: {accuracy_A_before:.2f}%")

# Learn Task B with no old examples in training
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
EPOCHS_B = 5
start = time.time()

for epoch in range(EPOCHS_B):
    model.train()
    total_loss = 0.0

    for images, labels in loader_B:
        # Convert labels 5-9 to 0-4 for the 5-output classifier
        labels = labels - 5
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Task B epoch {epoch + 1}/{EPOCHS_B} | "
        f"Loss: {total_loss / len(train_B):.4f}"
    )

# Evaluate on both tasks after learning B
accuracy_A_after = evaluate(model, test_loader_A)
accuracy_B_after = evaluate(model, test_loader_B)
forgetting = accuracy_A_before - accuracy_A_after

elapsed = time.time() - start

print("\n--- EXPERIMENT 2 RESULTS ---")
print(f"Task A accuracy before B: {accuracy_A_before:.2f}%")
print(f"Task A accuracy after B:  {accuracy_A_after:.2f}%")
print(f"Task B accuracy after B:  {accuracy_B_after:.2f}%")
print(f"Task A accuracy change:   {forgetting:.2f} percentage points")
print(f"Task B training time:    {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp2_sequential_no_replay.pt")

with open("/content/exp2_results.txt", "w") as f:
    f.write(f"Task A before B: {accuracy_A_before:.2f}%\n")
    f.write(f"Task A after B: {accuracy_A_after:.2f}%\n")
    f.write(f"Task B after B: {accuracy_B_after:.2f}%\n")
    f.write(f"Task A accuracy change: {forgetting:.2f} percentage points\n")
    f.write(f"Task B training time: {elapsed:.1f} seconds\n")

print("Experiment 2 checkpoint and results saved.")


Task A accuracy before Task B: 63.40%
Task B epoch 1/5 | Loss: 1.3339
Task B epoch 2/5 | Loss: 0.8772
Task B epoch 3/5 | Loss: 0.7208
Task B epoch 4/5 | Loss: 0.6223
Task B epoch 5/5 | Loss: 0.5215

--- EXPERIMENT 2 RESULTS ---
Task A accuracy before B: 63.40%
Task A accuracy after B:  10.00%
Task B accuracy after B:  74.00%
Task A accuracy change:   53.40 percentage points
Task B training time:    12.7 seconds
Experiment 2 checkpoint and results saved.


In [ ]:

# EXPERIMENT 2 — Corrected sequential learning
# A fixed 10-class output space prevents label remapping.

import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_A = choose_indices(train_data, TASK_A, 500, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def loader(dataset, ids, shuffle=False):
    return DataLoader(
        Subset(dataset, ids),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

loader_A = loader(train_data, train_A, True)
loader_B = loader(train_data, train_B, True)
test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def train_task(model, train_loader, epochs, task_name):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)  # Original labels 0–9

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)

        print(
            f"{task_name} epoch {epoch + 1}/{epochs} | "
            f"Loss: {total_loss / len(train_loader.dataset):.4f}"
        )

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))

            # Evaluate among the classes belonging to this task.
            # This measures task-aware classification, not global
            # 10-class class-incremental accuracy.
            task_scores = outputs[:, allowed_classes]
            predictions = torch.tensor(
                allowed_classes
            )[task_scores.argmax(dim=1)]

            correct += (predictions == labels).sum().item()
            total += labels.numel()

    return 100.0 * correct / total

model = SmallCNN10().to(device)
start = time.time()

# Stage 1: Learn Task A
train_task(model, loader_A, 5, "Task A")
acc_A_before = evaluate_task(model, test_loader_A, TASK_A)
print(f"\nTask A accuracy after Task A training: {acc_A_before:.2f}%")

# Save a checkpoint before learning Task B
torch.save(model.state_dict(), "/content/exp2_corrected_after_A.pt")

# Stage 2: Learn Task B without replaying Task A
train_task(model, loader_B, 5, "Task B")

acc_A_after = evaluate_task(model, test_loader_A, TASK_A)
acc_B_after = evaluate_task(model, test_loader_B, TASK_B)
forgetting = acc_A_before - acc_A_after
elapsed = time.time() - start

print("\n--- EXPERIMENT 2 CORRECTED RESULTS ---")
print(f"Task A before B: {acc_A_before:.2f}%")
print(f"Task A after B:  {acc_A_after:.2f}%")
print(f"Task B after B:  {acc_B_after:.2f}%")
print(f"Task A decline:  {forgetting:.2f} percentage points")
print(f"Total training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp2_corrected_final.pt")

with open("/content/exp2_corrected_results.txt", "w") as f:
    f.write("Fixed 10-output classifier; task-aware evaluation\n")
    f.write(f"Task A before B: {acc_A_before:.2f}%\n")
    f.write(f"Task A after B: {acc_A_after:.2f}%\n")
    f.write(f"Task B after B: {acc_B_after:.2f}%\n")
    f.write(f"Task A decline: {forgetting:.2f} percentage points\n")
    f.write(f"Total training time: {elapsed:.1f} seconds\n")

print("Corrected checkpoint and results saved.")


Task A epoch 1/5 | Loss: 1.3759
Task A epoch 2/5 | Loss: 1.0693
Task A epoch 3/5 | Loss: 0.9320
Task A epoch 4/5 | Loss: 0.8547
Task A epoch 5/5 | Loss: 0.7994

Task A accuracy after Task A training: 65.00%
Task B epoch 1/5 | Loss: 2.6970
Task B epoch 2/5 | Loss: 1.3942
Task B epoch 3/5 | Loss: 1.0488
Task B epoch 4/5 | Loss: 0.9103
Task B epoch 5/5 | Loss: 0.7864

--- EXPERIMENT 2 CORRECTED RESULTS ---
Task A before B: 65.00%
Task A after B:  26.60%
Task B after B:  67.80%
Task A decline:  38.40 percentage points
Total training time: 24.4 seconds
Corrected checkpoint and results saved.


In [ ]:

# EXPERIMENT 3: Experience replay
# Compare against corrected Experiment 2.

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

# Reuse the same dataset and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_A = choose_indices(train_data, TASK_A, 500, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def loader(dataset, indices, shuffle=False):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

# Same architecture as corrected Experiment 2
class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            predictions = outputs[:, allowed_classes].argmax(dim=1)
            predictions = torch.tensor(allowed_classes)[predictions]
            correct += (predictions == labels).sum().item()
            total += labels.numel()

    return 100.0 * correct / total

# Load the same Task A checkpoint used by Experiment 2
checkpoint = "/content/exp2_corrected_after_A.pt"
model = SmallCNN10().to(device)
model.load_state_dict(
    torch.load(checkpoint, map_location=device, weights_only=True)
)

# Deterministic memory: 50 examples per Task A class = 250 examples
memory_indices = choose_indices(train_data, TASK_A, 50, SEED + 1)

# Combine new Task B data with the small old-task memory
combined_dataset = ConcatDataset([
    Subset(train_data, train_B),
    Subset(train_data, memory_indices)
])

combined_loader = DataLoader(
    combined_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

print("Task B training images:", len(train_B))
print("Replay memory images:", len(memory_indices))
print("Combined training images:", len(combined_dataset))

# Evaluate Task A before replay training
acc_A_before = evaluate_task(model, test_loader_A, TASK_A)
print(f"Task A accuracy before B: {acc_A_before:.2f}%")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
EPOCHS = 5
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in combined_loader:
        images = images.to(device)
        labels = labels.to(device)  # Keep original labels 0–9

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Replay epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(combined_dataset):.4f}"
    )

acc_A_after = evaluate_task(model, test_loader_A, TASK_A)
acc_B_after = evaluate_task(model, test_loader_B, TASK_B)
forgetting = acc_A_before - acc_A_after
elapsed = time.time() - start

print("\n--- EXPERIMENT 3 RESULTS ---")
print(f"Task A before B: {acc_A_before:.2f}%")
print(f"Task A after replay: {acc_A_after:.2f}%")
print(f"Task B after replay: {acc_B_after:.2f}%")
print(f"Task A decline: {forgetting:.2f} percentage points")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp3_replay.pt")

with open("/content/exp3_replay_results.txt", "w") as f:
    f.write("Method: Experience replay\n")
    f.write("Memory size: 250 images\n")
    f.write(f"Task A before B: {acc_A_before:.2f}%\n")
    f.write(f"Task A after replay: {acc_A_after:.2f}%\n")
    f.write(f"Task B after replay: {acc_B_after:.2f}%\n")
    f.write(f"Task A decline: {forgetting:.2f} percentage points\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Experiment 3 checkpoint and results saved.")


Task B training images: 2500
Replay memory images: 250
Combined training images: 2750
Task A accuracy before B: 65.00%
Replay epoch 1/5 | Loss: 2.6238
Replay epoch 2/5 | Loss: 1.6530
Replay epoch 3/5 | Loss: 1.3193
Replay epoch 4/5 | Loss: 1.1228
Replay epoch 5/5 | Loss: 0.9777

--- EXPERIMENT 3 RESULTS ---
Task A before B: 65.00%
Task A after replay: 56.20%
Task B after replay: 71.80%
Task A decline: 8.80 percentage points
Training time: 13.7 seconds
Experiment 3 checkpoint and results saved.
